# WO8d — Environment↔culture correspondence: the high-gods look (EA034, exploratory)

**Exploratory, not confirmatory.** No predicted result, no null-hypothesis floor. The question is
"is there an environmental thread among the EA034 **'active, but not supporting morality'** societies
(n=42 in the full EA corpus; n=40 basin-joined), and is it more than shared ancestry" — not a
generalization claim about high-gods and environment.

Two corrections to the confirmatory frame that shape this notebook: **language family is labeled, not
nulled out** (three cousins clustering with several non-cousins is *two* findings — transmission and
convergence — not one residual), and the whole-sample backdrop makes "tight"/"distinctive" measurable
(tight *relative to what*).

**Query grammar: set-first**, not anchored — a trait filter in, a property of the resulting set
(cohesion, per lens, family-colored) out. Not CITYKIN (anchored retrieval) or TRACE (confirmatory
PERMANOVA) — see the WO's own scoping note.

**Engine:** `scripts/cdop/distance_core.py` (factored, not yet named a shared core — this is its first
real consumer; `tests/cdop/test_distance_core.py`, 10 green). Reuses the WO8b/8c drop-to-representative
metric and WO8c's point-window terrain lens; no new engine/API/UI.

**Accept gate** is not a verdict on high-gods — it is a legible whole-signature cohesion answer, with
magnitude reported alongside rank always, the whole-sample backdrop in place, family coloring in place,
and the Hopi check reported. A clean negative (the 42 scatter, no thread) passes.

WO: `docs/cdop/pilot/wo8d_env-culture-highgods.md`.

In [1]:
# Cell 1
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

import scripts.shared.db_utils as db_utils
from scripts.cdop.distance_core import (
    LENSES, backdrop_z, pairwise_distance, cohesion,
    random_draw_cohesions, family_restricted_draw_cohesions, percentile_rank,
)

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'cdop'
FOCUS_CODE = 'Active, but not supporting morality'   # EA034-3, n=42 in full EA corpus (Karl's selection)

print(f"ready | distance_core loaded | wo8c substrate present: {(OUT / 'wo8c_substrate.parquet').exists()}")

ready | distance_core loaded | wo8c substrate present: True


In [2]:
# Cell 2 -- Part A.1: EA034 already joined in wo8a's substrate build (`ea034_religion`, verified
# against the CLDF codebook directly: EA034 codes are Absent/Otiose/Active-not-supporting-morality/
# Active-supporting-morality, focus class n=42 in the full 1,291-society EA corpus -- matches the WO).
# No re-join needed. family_id crosswalk (WO4 method, 92.6% corpus-wide) already present too.
sub = pd.read_parquet(OUT / 'wo8c_substrate.parquet')
sub['is_focus'] = sub['ea034_religion'] == FOCUS_CODE

vc = sub['ea034_religion'].value_counts(dropna=False)
n_focus = int(sub['is_focus'].sum())
focus_fam_resolved = int(sub.loc[sub['is_focus'], 'family_id'].notna().sum())

print("\n".join([
    f"substrate societies (basin-joined, from wo8c): {len(sub)}",
    "",
    "EA034 distribution (basin-joined):",
    vc.to_string(),
    "",
    f"focus class '{FOCUS_CODE}': n={n_focus} basin-joined "
    f"(WO's full-corpus n=42; 2 not basin-joined at L08)",
    f"focus class family-resolved: {focus_fam_resolved} / {n_focus}  "
    f"(corpus-wide crosswalk rate: 92.6%, WO4)",
]))

substrate societies (basin-joined, from wo8c): 1133

EA034 distribution (basin-joined):
ea034_religion
None                                   453
Otiose                                 238
Absent                                 218
Active, supporting morality            184
Active, but not supporting morality     40

focus class 'Active, but not supporting morality': n=40 basin-joined (WO's full-corpus n=42; 2 not basin-joined at L08)
focus class family-resolved: 38 / 40  (corpus-wide crosswalk rate: 92.6%, WO4)


In [3]:
# Cell 3 -- Part A.2: derive ari_log (carried from WO8b/8c) and sanity-check the factored distance
# module's per-lens backdrop sizes against the real substrate before building anything on it.
sub['ari_log'] = np.log1p(sub['ari_ix_sav'])

lines = ["per-lens backdrop size (complete-case for that lens's columns):"]
for lens, cols in LENSES.items():
    ok, Xz = backdrop_z(sub, lens)
    n_focus_in_lens = int(ok['is_focus'].sum())
    lines.append(f"  {lens:8}{cols!s:45}n_backdrop={len(ok):5}  n_focus={n_focus_in_lens}")
print("\n".join(lines))

per-lens backdrop size (complete-case for that lens's columns):
  water   ['ari_log']                                  n_backdrop= 1133  n_focus=40
  thermal ['temperature_annual', 'tmp_seas_amp']       n_backdrop= 1133  n_focus=40
  overall ['ari_log', 'temperature_annual', 'tmp_seas_amp']n_backdrop= 1133  n_focus=40
  terrain ['relief_range_m', 'landform_position']      n_backdrop= 1124  n_focus=40


In [4]:
# Cell 4 -- Part B: whole-sample PCoA over the 'overall' (drop-to-representative) distance.
# Euclidean distance on already-standardized coordinates -> classical PCoA reduces to PCA on those
# coordinates directly (no Gower double-centering needed). Backdrop = grey cloud; the 42 = family-
# colored (families with >=2 members among the 42 get a distinct color -- that's the only case where
# "same color clustered together" can mean anything; singleton-family/unresolved members get a
# neutral marker, distinct from the grey backdrop but not implying any transmission reading).
ok, Xz = backdrop_z(sub, 'overall')
Vars = LENSES['overall']

U, S, Vt = np.linalg.svd(Xz, full_matrices=False)
scores = Xz @ Vt.T
var_explained = (S ** 2) / (S ** 2).sum()

focus_mask = ok['is_focus'].to_numpy()
focus = ok[focus_mask]
fam_counts = focus['family_id'].value_counts()
multi_fams = fam_counts[fam_counts >= 2].index.tolist()
cmap = plt.get_cmap('tab20')
fam_color = {fam: cmap(i % 20) for i, fam in enumerate(multi_fams)}

print("drawing WO8d Part B whole-sample PCoA (overall lens)...")
fig, ax = plt.subplots(figsize=(9, 8))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')

ax.scatter(scores[~focus_mask, 0], scores[~focus_mask, 1], s=8, c='lightgrey', alpha=0.5,
           linewidths=0, label=f'backdrop (n={(~focus_mask).sum()})')

singleton_idx = focus.index[~focus['family_id'].isin(multi_fams)]

ax.scatter(scores[singleton_idx, 0], scores[singleton_idx, 1], s=55, facecolors='none',
           edgecolors='black', linewidths=1.2, label='focus, singleton/unresolved family')
for fam in multi_fams:
    idx = focus.index[focus['family_id'] == fam]
    ax.scatter(scores[idx, 0], scores[idx, 1], s=70, color=fam_color[fam],
               edgecolors='black', linewidths=0.6, label=f'family {fam} (n={len(idx)})')

ax.set_xlabel(f"PC1 ({100*var_explained[0]:.0f}% var)", color='black')
ax.set_ylabel(f"PC2 ({100*var_explained[1]:.0f}% var)", color='black')
ax.set_title("WO8d Part B -- whole-sample PCoA, 'overall' (drop-to-representative) lens\n"
              "grey = backdrop; colored = focus class, same color = same language family (n>=2)",
              color='black')
ax.tick_params(colors='black')
for spine in ax.spines.values():
    spine.set_color('black')
ax.legend(fontsize=7, loc='best', framealpha=0.9)

outpath = OUT / 'wo8d_pcoa_overall.png'
fig.savefig(outpath, facecolor='white', dpi=140)
plt.close(fig)
display(IPImage(str(outpath)))

loadings = pd.DataFrame(Vt[:2].T, index=Vars, columns=['PC1', 'PC2'])
print("\n".join([
    f"variance explained: PC1={100*var_explained[0]:.1f}%  PC2={100*var_explained[1]:.1f}%  "
    f"PC3={100*var_explained[2]:.1f}%",
    "",
    "named-axis loadings (which raw variable each PC tracks):",
    loadings.round(3).to_string(),
    "",
    f"multi-member families among the {len(focus)} focus societies: {len(multi_fams)} "
    f"(covering {sum(fam_counts[f] for f in multi_fams)} societies); "
    f"{len(singleton_idx)} singleton-family/unresolved.",
]))

variance explained: PC1=61.2%  PC2=35.1%  PC3=3.6%

named-axis loadings (which raw variable each PC tracks):
                      PC1    PC2
ari_log             0.108 -0.960
temperature_annual  0.692  0.260
tmp_seas_amp       -0.714  0.107

multi-member families among the 40 focus societies: 5 (covering 26 societies); 14 singleton-family/unresolved.


In [5]:
# Cell 5 -- Part B addendum: hierarchical clustering over the SAME 'overall' distance, restricted to
# the 42, cut at 2 and 3 levels -- a caption over the continuous ordination above, never the primary
# claim. Extended (Karl, 2026-07-27) beyond the aggregate purity score to NAME who's in each cluster --
# the actual interest isn't just "how much is family," it's distinguishing three readings per knot:
#   (a) one family dominates      -> vertical transmission (cousins, expected, not the interesting part)
#   (b) 2+ DIFFERENT families mix -> candidate areal diffusion (borrowing across family lines, not
#                                     shared descent) -- this is the "secondary diffusion" signature
#   (c) true singletons, unclustered or clustered with unrelated members -> independent convergence
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

Xz_focus = Xz[focus_mask]                 # standardized 'overall' coords, focus rows only (rotation-
D_focus = pairwise_distance(Xz_focus)     # invariant: same pairwise distances as in `scores` space)
condensed = squareform(D_focus, checks=False)
Z = linkage(condensed, method='average')

cut2 = fcluster(Z, t=2, criterion='maxclust')
cut3 = fcluster(Z, t=3, criterion='maxclust')
stable = pd.crosstab(cut2, cut3)

focus_named = focus[['soc_id', 'name', 'family_id']].copy().reset_index(drop=True)
# NB: fillna is for DISPLAY only -- singleton/unresolved status below is decided against `multi_fams`
# (Cell 4, computed on the real family_id pre-fillna), never against a count of the literal string
# '(unresolved)' -- two unresolved societies are NOT the same family just because they share a label.
is_multi_fam = focus_named['family_id'].isin(multi_fams)
focus_named['family_id'] = focus_named['family_id'].fillna('(unresolved)')
focus_named['cut2'] = cut2
focus_named['cut3'] = cut3

cross2 = pd.crosstab(cut2, focus_named['family_id'])
max_share2 = (cross2.max(axis=1) / cross2.sum(axis=1)).round(2)
n_distinct_fam_per_cluster = (cross2 > 0).sum(axis=1)

lines = [
    "2-cut vs 3-cut stability (rows=2-cut, cols=3-cut; a stable 2-cut splits cleanly into 3-cut groups):",
    stable.to_string(), "",
    "2-cut cluster summary:",
    pd.DataFrame({'n': cross2.sum(axis=1), 'distinct_families': n_distinct_fam_per_cluster,
                  'max_family_share': max_share2}).to_string(),
    "",
    "FULL MEMBERSHIP by 2-cut cluster (name, family) -- read directly for (a) vs (b) vs (c) above:",
]
for c in sorted(focus_named['cut2'].unique()):
    members = focus_named[focus_named['cut2'] == c][['soc_id', 'name', 'family_id']]
    named_fams = members.loc[members['family_id'] != '(unresolved)', 'family_id']
    n_fam = named_fams.nunique()
    if n_fam == 0:
        tag = "no resolved family in this cluster"
    elif n_fam == 1:
        tag = "SINGLE-FAMILY (transmission)"
    else:
        tag = "MIXED FAMILIES (diffusion-across-lines candidate)"
    lines.append(f"\ncluster {c} (n={len(members)}, {n_fam} distinct NAMED family/families) -- {tag}:")
    lines.append(members.to_string(index=False))

singletons = focus_named[~is_multi_fam]
lines += ["", f"singleton-family / unresolved societies (n={len(singletons)}), named, with cluster "
              f"assignment (which knot they fell into, if any):",
          singletons[['soc_id', 'name', 'family_id', 'cut2', 'cut3']].to_string(index=False)]
print("\n".join(lines))

2-cut vs 3-cut stability (rows=2-cut, cols=3-cut; a stable 2-cut splits cleanly into 3-cut groups):
col_0  1   2  3
row_0          
1      3   0  0
2      0  31  6

2-cut cluster summary:
        n  distinct_families  max_family_share
row_0                                         
1       3                  3              0.33
2      37                 15              0.41

FULL MEMBERSHIP by 2-cut cluster (name, family) -- read directly for (a) vs (b) vs (c) above:

cluster 1 (n=3, 3 distinct NAMED family/families) -- MIXED FAMILIES (diffusion-across-lines candidate):
soc_id           name family_id
   Ec3        Chukchi  chuk1271
   Ec2          Yakut  turk1311
   Ec4 Yurak-Samoyeds  ural1272

cluster 2 (n=37, 14 distinct NAMED family/families) -- MIXED FAMILIES (diffusion-across-lines candidate):
soc_id          name    family_id
   Sb6         Wayuu     araw1281
   Sg2       Mapuche (unresolved)
   Sa5        Bribri     chib1249
   Nj2         Aztec     utoa1244
  Nb39      Sinkyon

In [7]:
# Cell 6 -- Part B addendum 2: does shared descent itself predict shared environment? Cell 5's raw
# membership already named the real structure without needing a finer cut: one family (atla1278,
# Atlantic-Congo) is 15/40 = 37.5% of the ENTIRE focus set (58% of everyone in any multi-member
# family) -- the single biggest fact for reading Part C. Two smaller real families (nilo1247 n=4,
# sino1245 n=3) are also visible directly. Plus the Siberian trio (Cell 5 cluster 1): three DIFFERENT
# families, tightest sub-group in the whole set -- a diffusion/convergence candidate, not transmission.
# Test each NAMED group's own cohesion vs random draws of the same size (same backdrop, 'overall' lens)
# -- does shared ancestry (atla1278, nilo1247, sino1245) also mean shared environment, or is family
# membership uninformative about environmental position even within one lineage?
NAMED_GROUPS = {
    'atla1278 (Atlantic-Congo, n=15)': ('family', 'atla1278'),
    'nilo1247 (Nilo-Saharan, n=4)':    ('family', 'nilo1247'),
    'sino1245 (Sino-Tibetan, n=3)':    ('family', 'sino1245'),
    'Siberian trio (3 diff. families)': ('soc_ids', ['Ec2', 'Ec3', 'Ec4']),
}

rows = []
for label, (kind, key) in NAMED_GROUPS.items():
    if kind == 'family':
        # Restrict to FOCUS-CLASS members of this family -- not every backdrop society with this
        # family_id (a bug in the first version of this cell: 'atla1278' alone has 289 members in the
        # whole ~1,133-society backdrop, not just the 15 who are also in the focus class).
        idx = ok.index[(ok['family_id'] == key) & focus_mask].tolist()
    else:
        idx = ok.index[ok['soc_id'].isin(key)].tolist()
    k = len(idx)
    obs = cohesion(Xz[idx])
    null = random_draw_cohesions(Xz, k=k, n_draws=2000, seed=0)
    rows.append({'group': label, 'n': k, 'obs_cohesion': obs,
                 'random_draw_mean': null.mean(), 'pct_tighter_than_random': 100 * percentile_rank(obs, null)})

named_result = pd.DataFrame(rows).set_index('group')
print("\n".join([
    "Does shared descent predict shared environment, within named groups ('overall' lens)?",
    named_result.round(3).to_string(),
    "",
    "Reading: high pct_tighter_than_random = this lineage/group IS environmentally coherent (descent",
    "tracks environment here); low/near-50% = shared ancestry does NOT predict shared environment for",
    "this group, even though they're genealogically related.",
]))

Does shared descent predict shared environment, within named groups ('overall' lens)?
                                   n  obs_cohesion  random_draw_mean  pct_tighter_than_random
group                                                                                        
atla1278 (Atlantic-Congo, n=15)   15         0.445             1.472                   100.00
nilo1247 (Nilo-Saharan, n=4)       4         0.564             1.318                    93.85
sino1245 (Sino-Tibetan, n=3)       3         1.291             1.228                    46.00
Siberian trio (3 diff. families)   3         0.987             1.228                    65.20

Reading: high pct_tighter_than_random = this lineage/group IS environmentally coherent (descent
tracks environment here); low/near-50% = shared ancestry does NOT predict shared environment for
this group, even though they're genealogically related.


In [8]:
# Cell 7 -- Part C: group cohesion, per lens, against the backdrop. The core quantitative output.
# No null-hypothesis verdict, no floor (WO's own proviso) -- this is descriptive context, "how unusual
# is this set," not a significance gate. Two baselines: fully-random (looser, primary per the WO --
# spatial dispersion already argues against pure ancestry) and family-restricted (stricter, "tighter
# than random cousins") -- both cheap, both reported, per-lens so a specific-lens lead is visible.
N_DRAWS = 2000
SEED = 0
rows = []
for lens in LENSES:
    ok_l, Xz_l = backdrop_z(sub, lens)
    fmask = ok_l['is_focus'].to_numpy()
    Xz_focus_l = Xz_l[fmask]
    obs = cohesion(Xz_focus_l)

    null_full = random_draw_cohesions(Xz_l, k=fmask.sum(), n_draws=N_DRAWS, seed=SEED)
    fam_backdrop = ok_l['family_id'].to_numpy()
    fam_focus = ok_l.loc[fmask, 'family_id'].to_numpy()
    null_fam = family_restricted_draw_cohesions(Xz_l, fam_backdrop, fam_focus, n_draws=N_DRAWS, seed=SEED)

    rows.append({
        'lens': lens, 'n_backdrop': len(ok_l), 'n_focus': int(fmask.sum()),
        'obs_cohesion': obs, 'random_draw_mean': null_full.mean(),
        'pct_tighter_than_random': 100 * percentile_rank(obs, null_full),
        'pct_tighter_than_cousins': 100 * percentile_rank(obs, null_fam),
    })

result = pd.DataFrame(rows).set_index('lens')
print("\n".join([
    "Cohesion (mean distance-to-centroid; LOWER = tighter) vs backdrop, all 4 lenses:",
    result.round(3).to_string(),
    "",
    "Reading: 'pct_tighter_than_random' = the group is tighter than this % of same-size random draws",
    "from the whole backdrop (the WO's primary, looser baseline). 'pct_tighter_than_cousins' = tighter",
    "than this % of draws that hold each member's language family fixed (the stricter baseline). Both",
    "are DESCRIPTIVE (how unusual is this set), not a significance test -- no floor, no verdict.",
]))

Cohesion (mean distance-to-centroid; LOWER = tighter) vs backdrop, all 4 lenses:
         n_backdrop  n_focus  obs_cohesion  random_draw_mean  pct_tighter_than_random  pct_tighter_than_cousins
lens                                                                                                           
water          1133       40         0.540             0.731                    95.25                     42.25
thermal        1133       40         1.223             1.201                    44.75                     59.85
overall        1133       40         1.441             1.525                    70.80                     52.40
terrain        1124       40         1.207             1.184                    44.10                     31.00

Reading: 'pct_tighter_than_random' = the group is tighter than this % of same-size random draws
from the whole backdrop (the WO's primary, looser baseline). 'pct_tighter_than_cousins' = tighter
than this % of draws that hold each member's languag

In [9]:
# Cell 8 -- Part D: the Hopi check. Sanity anchor, not validation (WO's own framing). NOTE: the Hopi
# (soc_id 'Nh18') are coded EA034='Absent' in this substrate -- NOT a member of the focus class (n=42,
# 'active but not supporting morality'). The rain-ritual intuition motivating this WO is about
# subsistence/ritual practice, not this specific EA034 code, so this is a general coherence check on
# the ordination/backdrop, not a check that Hopi is "in" the group being tested.
HOPI_ID = 'Nh18'
hopi_row = ok[ok['soc_id'] == HOPI_ID]
if hopi_row.empty:
    print(f"Hopi ({HOPI_ID}) not present in the 'overall'-lens backdrop -- cannot run the check.")
else:
    hopi_idx = hopi_row.index[0]
    hopi_fam = ok.loc[hopi_idx, 'family_id']
    d_from_hopi = np.sqrt(((Xz - Xz[hopi_idx]) ** 2).sum(axis=1))
    order = np.argsort(d_from_hopi)
    order = order[order != hopi_idx][:10]                  # 10 nearest neighbours, excluding self
    neighbours = ok.loc[order, ['soc_id', 'name', 'family_id']].copy()
    neighbours['dist'] = d_from_hopi[order]
    n_distinct_fam = neighbours['family_id'].nunique(dropna=True)
    n_same_fam = int((neighbours['family_id'] == hopi_fam).sum())

    tightest_lens = result['pct_tighter_than_random'].idxmax()
    lines = [
        f"Hopi ({HOPI_ID}): EA034 = '{sub.loc[sub['soc_id']==HOPI_ID, 'ea034_religion'].iloc[0]}' "
        f"-- NOT in the focus class. family_id = {hopi_fam}.",
        "",
        "10 nearest neighbours in the whole-sample 'overall' PCoA space:",
        neighbours.round(3).to_string(index=False),
        "",
        f"distinct families among Hopi's 10 nearest neighbours: {n_distinct_fam} / 10  "
        f"({n_same_fam} share Hopi's own family {hopi_fam})",
        "-> " + ("family-diverse neighbourhood (environment-driven knot reading is plausible)"
                 if n_distinct_fam >= 6 else
                 "neighbourhood dominated by few families (family-knot reading more plausible)"),
        "",
        f"Tightest lens for the 42 (Part C, highest pct_tighter_than_random): '{tightest_lens}'"
        + ("  -- matches the water/seasonality rain-ritual intuition"
           if tightest_lens in ('water', 'thermal') else
           "  -- does NOT match the water/seasonality rain-ritual intuition"),
    ]
    print("\n".join(lines))

Hopi (Nh18): EA034 = 'Absent' -- NOT in the focus class. family_id = utoa1244.

10 nearest neighbours in the whole-sample 'overall' PCoA space:
soc_id                                name family_id  dist
  Nd56                            San Juan  utoa1244 0.000
   Nh2                                Hano  kiow1265 0.000
  Nd49                        Antarianunts  utoa1244 0.208
   Nh3                              Navajo  atha1245 0.208
  Nd34                       Lida Shoshoni  utoa1244 0.255
  Nd33                              Beatty  utoa1244 0.282
  Nd27                         Kuyuidokado  utoa1244 0.296
  Nd30                        Eastern Mono  utoa1244 0.304
  Nd29 Tunava (Deep Springs and Fish Lake)  utoa1244 0.304
  Nd25                         Sawakudokwa  utoa1244 0.333

distinct families among Hopi's 10 nearest neighbours: 3 / 10  (8 share Hopi's own family utoa1244)
-> neighbourhood dominated by few families (family-knot reading more plausible)

Tightest lens for the 42 (

## Accept gate — pending Karl's run

Not filled in — written from actual Cell 2–8 output after Karl runs the notebook cell by cell and
reports back, not before (the arc's standing process rule: no number stated as a finding until read off
real output).

Checklist the gate requires (WO8d § Accept gate): a legible whole-signature answer to whether the 42 are
environmentally coherent (Cell 7), on which lenses (Cell 7, per-lens breakdown), and whether any
coherence reads as transmission or convergence (Cell 4/5/6, family coloring + named-cluster membership +
per-lineage cohesion); magnitude reported alongside rank always (Cell 7's `obs_cohesion`/
`random_draw_mean` columns beside the percentile columns); the whole-sample backdrop in place (Cell 3);
family coloring in place (Cell 4); and the Hopi check reported (Cell 8). A clean negative (the 42
scatter, no thread on any lens) passes the gate — it is a legitimate, likely outcome, not a failure.